# ASL Fingerspelling with a BERT-style Model (Kaggle)
This notebook converts per-frame hand landmarks into discrete **tokens** via **K-Means**,
then trains a compact **BERT** classifier using Hugging Face `transformers`.

> Works on the preprocessed landmarks dataset. Adjust paths if yours differ.

In [5]:
!pip -q install transformers==4.43.4 einops==0.8.0 torchmetrics==1.4.1 scikit-learn==1.5.1

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      + meson setup C:\Users\priyy\AppData\Local\Temp\pip-install-mmgot500\scikit-learn_137db77be4ba486aa4ccef0ca0f597db C:\Users\priyy\AppData\Local\Temp\pip-install-mmgot500\scikit-learn_137db77be4ba486aa4ccef0ca0f597db\.mesonpy-csng54hv -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\priyy\AppData\Local\Temp\pip-install-mmgot500\scikit-learn_137db77be4ba486aa4ccef0ca0f597db\.mesonpy-csng54hv\meson-python-native-file.ini
      The Meson build system
      Version: 1.8.3
      Source dir: C:\Users\priyy\AppData\Local\Temp\pip-install-mmgot500\scikit-learn_137db77be4ba486aa4ccef0ca0f597db
      Build dir: C:\Users\priyy\AppData\Local\Temp\pip-install-mmgot500\scikit-learn_137db77be4ba486aa4ccef0ca0f597db\.mesonpy-csng54hv
      Build type: native build
      Project name: scikit-learn
      Project version: 

In [6]:
import os, sys, math, random, gc, json, glob, re, time
from dataclasses import dataclass
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import torch
from torch.utils.data import Dataset

from transformers import (
    BertConfig, BertForSequenceClassification, Trainer, TrainingArguments
)

from torchmetrics.classification import MulticlassAccuracy, MulticlassF1Score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---- PATHS (edit if needed) ----
LANDMARK_DIR = "/kaggle/input/asl-fingerspelling-preprocessing-train-dataset/train_landmarks_npy"
CSV_CANDIDATES = [
    "/kaggle/input/asl-fingerspelling-preprocessing-train-dataset/train.csv",
    "/kaggle/input/asl-fingerspelling-preprocessing-train-dataset/labels.csv"
]

# ---- MODEL / TOKENIZATION PARAMS ----
PCA_DIM = 16      # dimensionality reduction for frames
VOCAB_SIZE = 512  # number of KMeans clusters (token vocab)
MAX_LEN   = 128   # max tokens per sample (truncate/pad)
BATCH_SIZE = 32
EPOCHS     = 5
LR         = 2e-4
WARMUP     = 0.1  # warmup ratio
WEIGHT_DECAY = 0.01

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


ModuleNotFoundError: No module named 'transformers'

In [ ]:
def discover_npy_files(root):
    paths = sorted(glob.glob(os.path.join(root, "**", "*.npy"), recursive=True))
    return paths

def try_read_labels_from_csv(candidates):
    for p in candidates:
        if os.path.exists(p):
            df = pd.read_csv(p)
            cols = {c.lower(): c for c in df.columns}
            fp_col = cols.get("filepath") or cols.get("file") or list(df.columns)[0]
            lb_col = cols.get("label") or list(df.columns)[1]
            df = df[[fp_col, lb_col]].rename(columns={fp_col:"filepath", lb_col:"label"})
            return df
    return None

def infer_label_from_path(path):
    parent = os.path.basename(os.path.dirname(path))
    m = re.search(r"label[=_-]([A-Za-z0-9]+)", os.path.basename(path))
    if m:
        return m.group(1)
    return parent

def build_index_df(landmark_dir):
    npy_files = discover_npy_files(landmark_dir)
    if len(npy_files) == 0:
        raise FileNotFoundError(f"No .npy files found under {landmark_dir}")
    csv_df = try_read_labels_from_csv(CSV_CANDIDATES)
    if csv_df is not None:
        base = os.path.dirname(landmark_dir)
        def resolve(p):
            if os.path.isabs(p) and os.path.exists(p):
                return p
            matches = [f for f in npy_files if os.path.basename(f) == os.path.basename(p)]
            if matches:
                return matches[0]
            cand = os.path.join(base, p.lstrip("/"))
            if os.path.exists(cand):
                return cand
            return p
        csv_df["filepath"] = csv_df["filepath"].apply(resolve)
        return csv_df
    else:
        rows = []
        for p in npy_files:
            rows.append({"filepath": p, "label": infer_label_from_path(p)})
        return pd.DataFrame(rows)

index_df = build_index_df(LANDMARK_DIR)
print("Total samples discovered:", len(index_df))
print(index_df.head())


In [ ]:
# Load a subset to fit PCA/KMeans (for speed). Adjust FRACTION if you like.
FRACTION = 0.2
subset_df = index_df.sample(frac=min(1.0, FRACTION), random_state=SEED) if len(index_df) > 2000 else index_df

def load_frames(paths, max_frames=500):
    feats = []
    count = 0
    for p in paths:
        arr = np.load(p)
        if arr.ndim != 2:
            arr = arr.reshape(arr.shape[0], -1)
        arr = arr - np.mean(arr, axis=0, keepdims=True)
        std = np.std(arr, axis=0, keepdims=True) + 1e-6
        arr = arr / std
        feats.append(arr)
        count += len(arr)
        if count >= max_frames:
            break
    if not feats:
        raise RuntimeError("No frames loaded to fit PCA/KMeans.")
    return np.concatenate(feats, axis=0)

fit_paths = subset_df["filepath"].tolist()
fit_frames = load_frames(fit_paths, max_frames=100000)
print("Frames for PCA/KMeans fit:", fit_frames.shape)

pca = PCA(n_components=16, random_state=SEED)
pca_feats = pca.fit_transform(fit_frames)
print("PCA reduced:", pca_feats.shape)

kmeans = KMeans(n_clusters=512, n_init=10, random_state=SEED, verbose=0)
kmeans.fit(pca_feats)
print("KMeans fitted with vocab size:", 512)


In [ ]:
PAD_ID = 512
CLS_ID = 513
VOCAB_TOTAL = 514

def frames_to_tokens(arr):
    if arr.ndim != 2:
        arr = arr.reshape(arr.shape[0], -1)
    arr = arr - np.mean(arr, axis=0, keepdims=True)
    std = np.std(arr, axis=0, keepdims=True) + 1e-6
    arr = arr / std
    z = pca.transform(arr)
    ids = kmeans.predict(z).astype(np.int64)
    return ids

def pack_tokens(token_ids, max_len=128, add_cls=True):
    if add_cls:
        token_ids = np.concatenate([[CLS_ID], token_ids], axis=0)
    if len(token_ids) > max_len:
        token_ids = token_ids[:max_len]
    attn = np.ones(len(token_ids), dtype=np.int64)
    if len(token_ids) < max_len:
        pad_len = max_len - len(token_ids)
        token_ids = np.concatenate([token_ids, np.full(pad_len, PAD_ID, dtype=np.int64)], axis=0)
        attn = np.concatenate([attn, np.zeros(pad_len, dtype=np.int64)], axis=0)
    return token_ids, attn


In [ ]:
class ASLBertDataset(Dataset):
    def __init__(self, df, label_encoder, max_len=128):
        self.df = df.reset_index(drop=True)
        self.le = label_encoder
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        arr = np.load(row.filepath)
        if arr.ndim != 2:
            arr = arr.reshape(arr.shape[0], -1)
        token_ids = frames_to_tokens(arr)
        input_ids, attention_mask = pack_tokens(token_ids, self.max_len, add_cls=True)
        label = self.le.transform([row.label])[0]
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(label, dtype=torch.long),
        }

labels = sorted(index_df["label"].unique())
le = LabelEncoder(); le.fit(labels)
num_classes = len(le.classes_)
print("Classes:", num_classes)
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(index_df, test_size=0.2, random_state=42, stratify=index_df["label"])
train_ds = ASLBertDataset(train_df, le, 128)
val_ds   = ASLBertDataset(val_df, le, 128)


In [ ]:
config = BertConfig(
    vocab_size=VOCAB_TOTAL,
    hidden_size=256,
    num_hidden_layers=6,
    num_attention_heads=8,
    intermediate_size=1024,
    max_position_embeddings=130,
    type_vocab_size=2,
    pad_token_id=PAD_ID,
    num_labels=num_classes,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
)
model = BertForSequenceClassification(config)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print('Params:', sum(p.numel() for p in model.parameters()))


In [ ]:
acc_metric = MulticlassAccuracy(num_classes=num_classes).to(device)
f1_metric  = MulticlassF1Score(num_classes=num_classes, average='macro').to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    with torch.no_grad():
        acc = acc_metric(torch.tensor(preds).to(device), torch.tensor(labels).to(device)).item()
        f1  = f1_metric(torch.tensor(preds).to(device), torch.tensor(labels).to(device)).item()
    acc_metric.reset(); f1_metric.reset()
    return {'accuracy': acc, 'macro_f1': f1}


In [ ]:
warmup_steps = int(0.1 * (len(train_ds) // 32) * 5)
args = TrainingArguments(
    output_dir='/kaggle/working/asl-bert',
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    learning_rate=2e-4,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    report_to='none',
    warmup_steps=warmup_steps,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=None,
    compute_metrics=compute_metrics,
)
trainer.train()
metrics = trainer.evaluate(); print(metrics)


In [ ]:
os.makedirs('/kaggle/working/asl-bert', exist_ok=True)
import pickle
with open('/kaggle/working/asl-bert/label_encoder.json','w') as f:
    json.dump({'classes': list(le.classes_)}, f)
with open('/kaggle/working/asl-bert/pca.pkl','wb') as f:
    pickle.dump(pca, f)
with open('/kaggle/working/asl-bert/kmeans.pkl','wb') as f:
    pickle.dump(kmeans, f)
with open('/kaggle/working/asl-bert/asl_bert_config.json','w') as f:
    json.dump({'PCA_DIM':16,'VOCAB_SIZE':512,'MAX_LEN':128,'BATCH_SIZE':32,'EPOCHS':5,'LR':2e-4}, f, indent=2)
print('Artifacts saved in /kaggle/working/asl-bert')
